# Día 3 — Clasificación: métricas, KNN y SVM

Del score a una decisión: cómo evaluar clasificadores con clases desbalanceadas.

## Objetivos de aprendizaje

- Distinguir Accuracy, Precision, Recall y F1 en contexto.
- Preparar datos mixtos sin fuga de información.
- Comparar KNN y SVM usando la misma validación.
- Explicar falsos positivos, falsos negativos y límites de generalización.

> **Tiempo del plan oficial:** 3 horas sincrónicas + 1 hora autónoma para métricas y clasificación. Prioriza baseline, KNN/SVM y lectura de errores; deja variaciones adicionales para el trabajo autónomo.

**Dataset:** Bank Marketing de UCI, guardado localmente en `datos/bank_marketing_uci.csv`; no se consulta UCI durante la ejecución. Fuente y licencia descritas en `datos/README.md`.

## Cómo estudiar esta jornada

Lee primero la pregunta y la explicación; ejecuta luego las celdas en orden. Después de cada tabla o gráfico, detente en **Interpreta** y redacta una conclusión con evidencia. Los resultados dependen de la versión de las librerías y de las particiones; usa tus propios números y no copies los ejemplos como verdades universales. No confundas predicción con causalidad.

## 1. El problema: identificar suscripciones sin esconder los errores

Usaremos **Bank Marketing**, datos públicos de campañas de una institución bancaria portuguesa. Cada fila es un contacto; `y` indica si terminó contratando un depósito. Es un caso de clasificación didáctico, no información del Banco Central del Ecuador ni evidencia sobre sus usuarios. El objetivo hoy es comparar KNN y SVM con un problema, una partición y métricas comunes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, average_precision_score,
                             precision_recall_curve)

ARCHIVO = DATA_DIR / "bank_marketing_uci.csv"
if not ARCHIVO.exists():
    raise FileNotFoundError(f"Falta {ARCHIVO}. Descarga el repositorio completo, incluida datos/.")
bank = pd.read_csv(ARCHIVO, sep=";")
print("Dimensiones:", bank.shape)
print("Columnas:", list(bank.columns))
bank.head()


### Interpreta los datos antes del modelo

La base oficial tiene 41 188 filas y 21 columnas. Las categorías `unknown` no son necesariamente errores: documentan información no conocida. `duration` es la duración de la llamada y solo se conoce después; la excluiremos porque el uso que planteamos decide **antes** de llamar. Sin esta decisión temporal, una métrica alta podría deberse a información que no existirá en producción.

In [ ]:
print("Distribución de y:")
print(bank["y"].value_counts())
print("Proporciones:")
print((bank["y"].value_counts(normalize=True) * 100).round(2))
print("Nulos por columna:")
print(bank.isna().sum().sort_values(ascending=False).head())
print("Categoría unknown:")
print(bank.select_dtypes("object").eq("unknown").sum().sort_values(ascending=False).head())


### Interpreta el desbalance

Aproximadamente 89 % son `no` y 11 % `yes`. Un clasificador que responda siempre `no` lograría cerca de 89 % de Accuracy y Recall de la clase positiva igual a cero. Por eso Accuracy no basta. Definimos `yes` como clase positiva: **Recall** mide qué proporción de suscriptores detectamos; **Precision**, qué proporción de los contactos marcados como posibles suscriptores lo fueron; **F1** equilibra ambas.

## 2. Holdout común y preparación sin fuga

Separamos Train, Validation y Test antes de aprender escalas o categorías. Estratificamos para conservar aproximadamente el porcentaje de `yes`. Test queda cerrado hasta que hayamos elegido el modelo.

In [ ]:
# Muestra estratificada reproducible para mantener las prácticas ágiles en un computador de clase.
datos = bank.groupby("y", group_keys=False).sample(frac=min(12000 / len(bank), 1), random_state=42)
y = datos["y"].map({"no": 0, "yes": 1})
# duration se excluye por fuga temporal; el resto debe estar disponible antes de la llamada.
X = datos.drop(columns=["y", "duration"])
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=.50, random_state=42, stratify=y_rest)
num = X.select_dtypes(include="number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
print("Train / Validation / Test:", X_train.shape, X_val.shape, X_test.shape)
print("Proporción yes en Train/Val/Test:", [round(float(s.mean()), 3) for s in (y_train, y_val, y_test)])


## 3. Métricas con significado operativo

Primero medimos un baseline de clase mayoritaria. Luego comparamos modelos sobre exactamente las mismas filas de Validation. Antes de ejecutar, anota qué error sería más costoso en el escenario supuesto; no existe una métrica «mejor» sin definir el costo de falsos positivos y falsos negativos.

In [ ]:
def resumen(y_real, y_pred, nombre):
    return {"modelo": nombre,
            "accuracy": accuracy_score(y_real, y_pred),
            "precision_yes": precision_score(y_real, y_pred, zero_division=0),
            "recall_yes": recall_score(y_real, y_pred, zero_division=0),
            "f1_yes": f1_score(y_real, y_pred, zero_division=0)}

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_base = baseline.predict(X_val)
pd.DataFrame([resumen(y_val, pred_base, "Siempre clase mayoritaria")]).set_index("modelo").round(3)


### Interpreta el baseline

La Accuracy del baseline debe aproximarse a la proporción de `no`, pero Recall de `yes` y F1 serán cero. Esta es la referencia mínima: cualquier árbol, KNN o SVM debe compararse con ella, especialmente en detección de `yes`. Si tu output difiere, comprueba la estratificación y qué clase codificaste como positiva.

## 4. K vecinos más cercanos (KNN)

KNN asigna la clase más frecuente entre las observaciones cercanas. Una variable con escala grande dominaría la distancia; por eso StandardScaler se ajusta **solo dentro de Train** en un Pipeline. Un `k` pequeño puede reaccionar al ruido; uno grande suaviza las fronteras y puede ignorar grupos minoritarios.

In [ ]:
resultados = [resumen(y_val, pred_base, "Baseline")]
modelos = {}
for k in [5, 15, 31]:
    modelo = Pipeline([("prep", pre), ("model", KNeighborsClassifier(n_neighbors=k, weights="distance"))])
    modelo.fit(X_train, y_train)
    modelos[f"KNN k={k}"] = modelo
    resultados.append(resumen(y_val, modelo.predict(X_val), f"KNN k={k}"))
tabla_knn = pd.DataFrame(resultados).set_index("modelo")
tabla_knn.round(3)


### Interpreta KNN

Compara Recall y F1, no solo Accuracy. Si al aumentar `k` cae Recall, el vecindario puede estar favoreciendo la clase mayoritaria. Si cambia poco, el resultado puede ser robusto a estos valores de `k`, pero tres configuraciones no bastan para afirmar estabilidad. El escalado protege la distancia; One-Hot hace que las categorías entren como indicadores, una decisión cuyo efecto también debe revisarse.

## 5. SVM: margen, kernel y escala

SVM busca una frontera con margen amplio. `C` controla cuánto penaliza errores de Train: valores grandes fuerzan menos errores de entrenamiento y pueden generar fronteras más ajustadas. Kernel lineal representa una frontera lineal; RBF permite fronteras curvas. `gamma` regula el alcance de cada observación en RBF. El escalado es esencial.

In [ ]:
for kernel, C in [("linear", 1), ("rbf", .5), ("rbf", 2)]:
    modelo = Pipeline([("prep", pre), ("model", SVC(kernel=kernel, C=C, gamma="scale", class_weight="balanced"))])
    modelo.fit(X_train, y_train)
    nombre = f"SVM {kernel}, C={C}"
    modelos[nombre] = modelo
    resultados.append(resumen(y_val, modelo.predict(X_val), nombre))
tabla = pd.DataFrame(resultados).drop_duplicates("modelo").set_index("modelo").sort_values("f1_yes", ascending=False)
tabla.round(3)


### Interpreta la comparación de modelos

Busca el compromiso entre Precision y Recall. `class_weight='balanced'` aumenta el peso de la clase minoritaria, pero no garantiza un resultado superior; puede elevar Recall a costa de más falsos positivos. Kernel RBF con `C` más alto puede ajustar fronteras más complejas, pero decide según Validation. En este diseño SVM se entrena sobre una muestra estratificada para ajustarse al tiempo de clase; reporta que el muestreo limita la comparación con entrenamientos de tamaño completo.

In [ ]:
# Elegimos F1 en Validation como regla de demostración; cámbiala si el costo del negocio exige priorizar Recall o Precision.
comparables = tabla.drop(index="Baseline", errors="ignore")
elegido = comparables["f1_yes"].idxmax()
modelo_final = modelos[elegido]
print("Elegido usando solo Validation:", elegido)
pred_val = modelo_final.predict(X_val)
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["no", "yes"], values_format="d")
plt.title(f"Matriz de confusión — {elegido} — Validation")
plt.show()
print(classification_report(y_val, pred_val, target_names=["no", "yes"], zero_division=0))


### Lee la matriz de confusión

Filas = clase real; columnas = clase predicha. Arriba a la derecha están los falsos positivos; abajo a la izquierda, los falsos negativos. Interpreta conteos absolutos y métricas juntas. Una mejora en Recall puede implicar más clientes contactados en vano; una Precision alta con Recall muy bajo puede dejar muchas suscripciones sin detectar.

## 6. Evaluación final, umbral y límites

El `predict()` de SVM usa una frontera predeterminada. En sistemas con scores probabilísticos se puede elegir otro umbral usando Validation; jamás se ajusta mirando Test. Este ejercicio mantiene el umbral predeterminado para concentrarse en KNN/SVM y métricas. El resultado final es una estimación en datos retenidos, no una promesa de desempeño futuro ni una recomendación institucional.

In [ ]:
pred_test = modelo_final.predict(X_test)
pd.DataFrame([resumen(y_val, pred_val, "Validation"), resumen(y_test, pred_test, "Test")]).set_index("modelo").round(3)


### Conclusión guiada

Escribe 4–6 frases: (1) qué modelo elegiste y con qué regla en Validation; (2) cuánto cambia F1/Recall entre Validation y Test; (3) cuántos falsos positivos y falsos negativos aparecen; (4) si mejora al baseline en la métrica de interés; (5) una limitación del dataset y una del diseño. Si escoges después de mirar Test, ya no es una evaluación independiente.

## 7. Actividad autónoma y comprobación de aprendizaje

Sin cambiar el Test: compara `k=5,15,31`; explica cómo cambia Precision/Recall. Compara SVM lineal y RBF. Construye una matriz de confusión y escribe qué error priorizarías reducir en un caso de alertas económicas o supervisión, explicitando que Bank Marketing no representa una población ecuatoriana.

**Autoevaluación:** ¿por qué se escala dentro del Pipeline? ¿Por qué Accuracy engaña con clases desbalanceadas? ¿Qué información excluimos por leakage? ¿Por qué Test se usa una sola vez?